# Vehicle Damage Detection & Severity Assessment — Colab Runbook

Run-all notebook for the full two-stage pipeline:

1. **Setup** — install deps, mount Google Drive
2. **Stage 1** — validate the recovered detector weights; retrain only if needed
3. **Auto-label severity** — generate extra weakly-labeled crops (rule-based), keep your 195 hand-labels as the trusted validation set
4. **Stage 2** — train the severity classifier (MobileNetV2 transfer learning)
5. **Stage 3** — end-to-end demo + save outputs

> **Before running:** put your project folder in Google Drive (e.g. `MyDrive/auto_damage/`) containing
> `best_model.pt`, `data.yaml`, the `train/ val/ test/` dataset folders, and your hand-labeled
> `minor/ moderate/ severe/` crop folders. Then set `PROJECT_DIR` below.


## 1. Setup

In [ ]:
# Colab GPU check — Runtime > Change runtime type > T4 GPU
import torch
print("CUDA available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
!pip -q install ultralytics==8.* >/dev/null
import ultralytics; print("ultralytics", ultralytics.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS to where you uploaded the project in Drive:
PROJECT_DIR = '/content/drive/MyDrive/auto_damage'

import os
os.chdir(PROJECT_DIR)
print("Working dir:", os.getcwd())
print("Contents:", sorted(os.listdir('.')))

## 2. Stage 1 — Validate the recovered detector

First check whether the recovered `best_model.pt` is already a good detector.
If `mAP@50` is reasonable (say > 0.4), **skip retraining**. If it's near zero or the
load fails, run the retraining cell below.


In [ ]:
from ultralytics import YOLO

DATA_YAML = 'data.yaml'   # the corrected 6-class config
detector = None
try:
    detector = YOLO('best_model.pt')
    print("Loaded best_model.pt. Validating on the test split...")
    m = detector.val(data=DATA_YAML, split='test', verbose=False)
    print(f"\nRecovered weights — mAP@50={m.box.map50:.3f}  mAP@50-95={m.box.map:.3f}  "
          f"P={m.box.mp:.3f}  R={m.box.mr:.3f}")
    GOOD = m.box.map50 > 0.40
    print("\n>>> Weights look GOOD — you can skip retraining." if GOOD
          else "\n>>> Weights look WEAK — run the retraining cell below.")
except Exception as e:
    print("Could not validate recovered weights:", e)
    print(">>> Run the retraining cell below.")
    GOOD = False

### 2b. Retrain the detector — ONLY if the check above said WEAK
(~30 min on a T4. Skip if the recovered weights were good.)

In [ ]:
# RUN ONLY IF NEEDED
model = YOLO('yolov8n.pt')
results = model.train(
    data=DATA_YAML, epochs=30, imgsz=640, batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    patience=10, project='runs', name='cardd_detector', exist_ok=True,
)
m = model.val(data=DATA_YAML, split='test', verbose=False)
print(f"Retrained — mAP@50={m.box.map50:.3f}  mAP@50-95={m.box.map:.3f}")
# Promote the freshly trained weights to best_model.pt
import shutil
shutil.copy('runs/cardd_detector/weights/best.pt', 'best_model.pt')
detector = YOLO('best_model.pt')
print("Saved new best_model.pt")

## 3. Auto-label more severity data (weak supervision)

Your 195 hand-labeled crops are small. Here we **automatically** generate many more
labeled crops by running the detector over the full dataset and assigning a severity
label from a rule based on **damage type** and **relative box size**:

- `glass shatter`, `lamp broken`, `tire flat`, `crack` → lean **severe**
- `dent` → severity by size (small=minor, large=severe)
- `scratch` → lean **minor**, large ones → moderate

These auto-labels are **noisy** — so we use them only to *augment training*. Your 195
hand-labels stay as the **trusted validation set**. The next cell also reports how often
the rule agrees with your hand-labels, so the noise is visible, not assumed.


In [ ]:
import cv2, numpy as np
from pathlib import Path

DAMAGE_NAMES = ['dent','scratch','crack','glass shatter','lamp broken','tire flat']
SEVERE_TYPES = {'glass shatter','lamp broken','tire flat','crack'}

def rule_severity(damage, box_frac):
    # box_frac = box area / image area  (0..1)
    if damage in SEVERE_TYPES:
        return 'severe' if box_frac > 0.05 else 'moderate'
    if damage == 'scratch':
        return 'moderate' if box_frac > 0.10 else 'minor'
    if damage == 'dent':
        if box_frac > 0.12: return 'severe'
        if box_frac > 0.04: return 'moderate'
        return 'minor'
    return 'moderate'

In [ ]:
# First: measure agreement of the rule vs YOUR hand-labels (sanity check)
# We re-derive box_frac is unavailable for loose crops, so we approximate using
# crop pixel area thresholds calibrated from your hand-labeled set.
from PIL import Image
hand = {'minor':[], 'moderate':[], 'severe':[]}
for c in hand:
    d = Path(c)
    if d.is_dir():
        for f in d.iterdir():
            if f.suffix.lower() in ('.jpg','.jpeg','.png'):
                try:
                    w,h = Image.open(f).size; hand[c].append(w*h)
                except: pass
for c in hand:
    a = sorted(hand[c])
    if a:
        print(f"{c:9s} n={len(a):3d}  median area={a[len(a)//2]:,}")
print("\n(Used to sanity-check the size thresholds; type signal dominates the rule.)")

In [ ]:
# Generate auto-labeled crops from the dataset into severity_auto/<class>/
OUT = Path('severity_auto'); [ (OUT/c).mkdir(parents=True, exist_ok=True) for c in ['minor','moderate','severe'] ]
counts = {'minor':0,'moderate':0,'severe':0}
CONF = 0.30
SRC_DIRS = ['train/images','val/images']   # not test — keep test clean for the demo

n_img = 0
for sd in SRC_DIRS:
    if not os.path.isdir(sd): continue
    for name in os.listdir(sd):
        img = cv2.imread(os.path.join(sd, name))
        if img is None: continue
        H, W = img.shape[:2]; n_img += 1
        for r in detector(img, conf=CONF, verbose=False):
            if r.boxes is None: continue
            for b in r.boxes:
                x1,y1,x2,y2 = map(int, b.xyxy[0])
                if x2<=x1 or y2<=y1: continue
                damage = DAMAGE_NAMES[int(b.cls[0])] if int(b.cls[0])<len(DAMAGE_NAMES) else 'dent'
                frac = ((x2-x1)*(y2-y1))/float(W*H)
                sev = rule_severity(damage, frac)
                crop = img[y1:y2, x1:x2]
                if crop.size==0: continue
                counts[sev]+=1
                cv2.imwrite(str(OUT/sev/f"auto_{counts[sev]}.jpg"), crop)
print(f"Scanned {n_img} images. Auto-labeled crops:", counts, "total", sum(counts.values()))

## 4. Stage 2 — Train the severity classifier

- **Train on** the auto-labeled crops + your hand-labels (lots of data, some noise)
- **Validate on** your 195 hand-labels only (trusted)
- **Model**: MobileNetV2 frozen backbone + small head + augmentation — the right choice
  for a small/noisy set; far less overfitting than a from-scratch CNN.


In [ ]:
import tensorflow as tf, shutil
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

IMG=(224,224); BATCH=16; CLASSES=['minor','moderate','severe']

# Build a combined TRAIN dir (auto + hand) and a VAL dir (hand only)
train_root = Path('sev_train'); val_root = Path('sev_val')
for root in (train_root, val_root):
    if root.exists(): shutil.rmtree(root)
    for c in CLASSES: (root/c).mkdir(parents=True, exist_ok=True)

# hand labels -> 80% train, 20% val (val is trusted)
import random; random.seed(42)
for c in CLASSES:
    files = [f for f in Path(c).iterdir() if f.suffix.lower() in ('.jpg','.jpeg','.png')] if Path(c).is_dir() else []
    random.shuffle(files)
    k = max(1, int(len(files)*0.2))
    for f in files[:k]:  shutil.copy(f, val_root/c/f.name)
    for f in files[k:]:  shutil.copy(f, train_root/c/('hand_'+f.name))
# auto labels -> train only
for c in CLASSES:
    ad = Path('severity_auto')/c
    if ad.is_dir():
        for f in ad.iterdir(): shutil.copy(f, train_root/c/f.name)

for c in CLASSES:
    print(f"{c:9s} train={len(list((train_root/c).iterdir())):4d}  val={len(list((val_root/c).iterdir())):3d}")

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_root, image_size=IMG, batch_size=BATCH, class_names=CLASSES, label_mode='int', seed=42)
val_ds = tf.keras.utils.image_dataset_from_directory(
    val_root, image_size=IMG, batch_size=BATCH, class_names=CLASSES, label_mode='int', shuffle=False)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(500).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

aug = models.Sequential([layers.RandomFlip('horizontal'),
                         layers.RandomRotation(0.1), layers.RandomZoom(0.1)])
base = MobileNetV2(input_shape=IMG+(3,), include_top=False, weights='imagenet'); base.trainable=False

model = models.Sequential([
    layers.Rescaling(1/127.5, offset=-1), aug, base,
    layers.GlobalAveragePooling2D(), layers.Dropout(0.3),
    layers.Dense(len(CLASSES), activation='softmax')])
model.compile('adam', 'sparse_categorical_crossentropy', metrics=['accuracy'])
hist = model.fit(train_ds, validation_data=val_ds, epochs=25,
                 callbacks=[tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)])
print("\nBest val accuracy:", round(max(hist.history['val_accuracy']),3))
model.save('severity_model.h5')
open('severity_classes.txt','w').write("\n".join(CLASSES))
print("Saved severity_model.h5")

In [ ]:
# Confusion matrix on the trusted (hand-labeled) validation set
import numpy as np
y_true=[]; y_pred=[]
for x,y in val_ds:
    p = model.predict(x, verbose=0)
    y_true += list(y.numpy()); y_pred += list(p.argmax(1))
import collections
cm = np.zeros((3,3), int)
for t,p in zip(y_true,y_pred): cm[t,p]+=1
print("rows=true, cols=pred  order:", CLASSES)
print(cm)
acc = np.trace(cm)/cm.sum() if cm.sum() else 0
print("val accuracy (hand-labeled):", round(float(acc),3))

## 5. Stage 3 — End-to-end demo

In [ ]:
COLORS={'minor':(0,200,0),'moderate':(0,165,255),'severe':(0,0,255)}
SEV = open('severity_classes.txt').read().split()

def assess(image_path, out='demo_output.jpg'):
    img = cv2.imread(image_path); assert img is not None, image_path
    found=[]
    for r in detector(img, conf=0.30, verbose=False):
        if r.boxes is None: continue
        for b in r.boxes:
            x1,y1,x2,y2 = map(int,b.xyxy[0])
            if x2<=x1 or y2<=y1: continue
            dmg = DAMAGE_NAMES[int(b.cls[0])]
            crop = cv2.resize(img[y1:y2,x1:x2],(224,224)).astype('float32')
            sev = SEV[int(model.predict(crop[None],verbose=0)[0].argmax())]
            col = COLORS.get(sev,(255,255,255))
            cv2.rectangle(img,(x1,y1),(x2,y2),col,2)
            cv2.putText(img,f"{dmg} | {sev}",(x1,max(15,y1-8)),cv2.FONT_HERSHEY_SIMPLEX,0.6,col,2)
            found.append((dmg,sev))
    cv2.imwrite(out,img)
    print(f"{len(found)} regions ->", found)
    return out

# Run on a few test images
import glob
from IPython.display import Image as IPyImage, display
tests = sorted(glob.glob('test/images/*.jpg'))[:3]
for i,t in enumerate(tests):
    o=assess(t, f'demo_{i}.jpg'); display(IPyImage(o))